# Context Features

Creates static and time-varying context features (amenities, transport, deprivation, broadband) and merges them into the property panel.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

try:
    import geopandas as gpd
    HAVE_GPD = True
except Exception:
    HAVE_GPD = False
    print("[NB4] GeoPandas unavailable; will only use fast non-spatial paths where possible.")

INTERIM = Path("../data/interim")
CONTEXT = (INTERIM / "context"); CONTEXT.mkdir(parents=True, exist_ok=True)

def month_endify(s):
    s = pd.to_datetime(s)
    return (s + pd.offsets.MonthEnd(0)).astype("datetime64[ns]")

def winsorize(x, p_low=0.01, p_high=0.99):
    x = pd.Series(x, dtype="float64")
    if x.empty or x.isna().all(): return x
    lo, hi = np.nanpercentile(x, [p_low*100, p_high*100])
    return x.clip(lower=lo, upper=hi)

def zscore(x):
    x = pd.Series(x, dtype="float64")
    m, s = x.mean(skipna=True), x.std(skipna=True)
    return (x - m) / s if s and not np.isnan(s) and s != 0 else x

def safe_rate(numer, denom, scale=1.0):
    return np.where(denom > 0, (numer / denom) * scale, np.nan)

# Base LA list 
panel_all_path = INTERIM / "la_month_panel_ALL.parquet"
assert panel_all_path.exists(), "Run NB3 through Cell 4 first (creates la_month_panel_ALL.parquet)."
panel_all = pd.read_parquet(panel_all_path, columns=["LA_code","LA_name"]).drop_duplicates("LA_code")

# Area & households (used for rates)
area_path = INTERIM / "la_centroids_area.parquet"
if area_path.exists():
    area = pd.read_parquet(area_path)
    if "area_km2" not in area.columns and "area_m2" in area.columns:
        area["area_km2"] = area["area_m2"] / 1_000_000.0
    area = area[["LA_code","area_km2"]].drop_duplicates("LA_code")
else:
    print("[NB4][WARN] la_centroids_area.parquet missing — density metrics may be skipped.")
    area = pd.DataFrame(columns=["LA_code","area_km2"])

hh_path = INTERIM / "households_la.parquet"
if hh_path.exists():
    hh = pd.read_parquet(hh_path)[["LA_code","households_2021"]].drop_duplicates("LA_code")
else:
    print("[NB4][WARN] households_la.parquet missing — per-household rates will be NaN.")
    hh = pd.DataFrame(columns=["LA_code","households_2021"])

la_base = (panel_all
           .merge(hh,   on="LA_code", how="left")
           .merge(area, on="LA_code", how="left"))
la_base["households_2021"] = la_base["households_2021"].astype("float64")


## Static Context Features

In [6]:
# POI to LA helpers 
def _guess_latlon(df: pd.DataFrame) -> tuple[str|None, str|None]:
    lat_candidates = ["lat", "latitude", "Lat", "LAT", "y", "Y"]
    lon_candidates = ["lon", "lng", "long", "longitude", "Lon", "LON", "x", "X"]
    lat = next((c for c in lat_candidates if c in df.columns), None)
    lon = next((c for c in lon_candidates if c in df.columns), None)
    return lat, lon

def load_la_boundaries():
    p = INTERIM / "la_boundaries.parquet"
    if not (HAVE_GPD and p.exists()):
        return None
    g = gpd.read_parquet(p)
    if "geometry" not in g.columns:
        print("[NB4] WARN: la_boundaries has no geometry; spatial join disabled.")
        return None
    return g.to_crs("EPSG:4326")

def poi_to_la_count(poi_path: Path, id_col: str|None=None,
                    la_code_col: str|None=None,
                    lat_col: str|None=None, lon_col: str|None=None,
                    label: str="(poi)") -> pd.DataFrame:
    """
    Prefer fast groupby by LA_code. Otherwise do spatial join (if GeoPandas + boundaries are available).
    Handles geometry columns (WKB) or lat/lon columns.
    Returns DataFrame [LA_code, count]. Logs what it did.
    """
    if not poi_path.exists():
        print(f"[NB4] SKIP {label}: {poi_path.name} not found.")
        return pd.DataFrame(columns=["LA_code","count"])

    # Try reading as GeoDataFrame first 
    if HAVE_GPD:
        try:
            gpoi = gpd.read_parquet(poi_path)
            if "geometry" in gpoi.columns and len(gpoi) > 0:
                
                if id_col is None:
                    id_col = next((c for c in gpoi.columns if c not in {"LA_code","geometry"}), gpoi.columns[0])

                # Fast path if LA_code already there
                if la_code_col and la_code_col in gpoi.columns:
                    out = (gpoi.groupby(la_code_col)[id_col].size()
                             .rename("count").reset_index()
                             .rename(columns={la_code_col:"LA_code"}))
                    print(f"[NB4] OK {label}: used LA_code in file → {len(out)} LAs, {int(out['count'].sum())} points.")
                    return out

                # Spatial join using geometry
                bounds = load_la_boundaries()
                if bounds is None:
                    print(f"[NB4] SKIP {label}: LA boundaries not available.")
                    return pd.DataFrame(columns=["LA_code","count"])

                # Ensure same CRS
                if gpoi.crs is None:
                    gpoi.set_crs("EPSG:4326", inplace=True)
                gpoi = gpoi.to_crs("EPSG:4326")
                bounds = bounds.to_crs("EPSG:4326")

                joined = gpd.sjoin(gpoi, bounds[["LA_code","geometry"]], how="left", predicate="within")
                out = (joined.dropna(subset=["LA_code"])
                       .groupby("LA_code")[id_col].size()
                       .rename("count").reset_index())
                print(f"[NB4] OK {label}: spatial join via geometry column → {len(out)} LAs, {int(out['count'].sum())} points.")
                return out
        except Exception as e:
            # Fall back to pandas if GeoDataFrame read fails
            pass

    # Fall back to pandas read
    df = pd.read_parquet(poi_path)
    if id_col is None:
        # pick a stable id-like column (first non-geo)
        id_col = next((c for c in df.columns if c not in {"LA_code","lat","lon","latitude","longitude","geometry"}), df.columns[0])

    # Fast path if LA_code already present
    if la_code_col and la_code_col in df.columns:
        out = (df.groupby(la_code_col)[id_col].size()
                 .rename("count").reset_index()
                 .rename(columns={la_code_col:"LA_code"}))
        print(f"[NB4] OK {label}: used LA_code in file → {len(out)} LAs, {int(out['count'].sum())} points.")
        return out

    # Spatial path if possible (using lat/lon)
    if not HAVE_GPD:
        print(f"[NB4] SKIP {label}: no LA_code and GeoPandas unavailable.")
        return pd.DataFrame(columns=["LA_code","count"])

    bounds = load_la_boundaries()
    if bounds is None:
        print(f"[NB4] SKIP {label}: LA boundaries not available.")
        return pd.DataFrame(columns=["LA_code","count"])

    # Guess lat/lon if not provided
    lat_col = lat_col or _guess_latlon(df)[0]
    lon_col = lon_col or _guess_latlon(df)[1]
    if lat_col is None or lon_col is None:
        print(f"[NB4] SKIP {label}: could not find lat/lon columns in {poi_path.name}.")
        return pd.DataFrame(columns=["LA_code","count"])

    gpoi = gpd.GeoDataFrame(df,
                            geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
                            crs="EPSG:4326")
    joined = gpd.sjoin(gpoi, bounds[["LA_code","geometry"]], how="left", predicate="within")
    out = (joined.dropna(subset=["LA_code"])
           .groupby("LA_code")[id_col].size()
           .rename("count").reset_index())
    print(f"[NB4] OK {label}: spatial join via {lat_col}/{lon_col} → {len(out)} LAs, {int(out['count'].sum())} points.")
    return out



## Time-Varying Context Features 

In [9]:
# STATIC context 
parts = []
built = []  # keep small status strings for summary

# Fast food per 1k households
ff_path = INTERIM / "amenities_fast_food.parquet"
if ff_path.exists():
    ff = pd.read_parquet(ff_path)
    la_col = "LA_code" if "LA_code" in ff.columns else None
    ff_cnt = poi_to_la_count(ff_path, id_col=ff.columns[0], la_code_col=la_col, label="fast_food")
    if not ff_cnt.empty:
        tmp = la_base.merge(ff_cnt, on="LA_code", how="left")
        tmp["fastfood_per_1k_hh"] = safe_rate(tmp["count"], tmp["households_2021"], 1000.0)
        parts.append(tmp[["LA_code","fastfood_per_1k_hh"]])
        built.append("fastfood_per_1k_hh")
else:
    print("[NB4] SKIP fast_food: file missing.")

# Supermarkets per 1k households
sm_path = INTERIM / "amenities_supermarket.parquet"
if sm_path.exists():
    sm = pd.read_parquet(sm_path)
    la_col = "LA_code" if "LA_code" in sm.columns else None
    sm_cnt = poi_to_la_count(sm_path, id_col=sm.columns[0], la_code_col=la_col, label="supermarkets")
    if not sm_cnt.empty:
        tmp = la_base.merge(sm_cnt, on="LA_code", how="left")
        tmp["supermkt_per_1k_hh"] = safe_rate(tmp["count"], tmp["households_2021"], 1000.0)
        parts.append(tmp[["LA_code","supermkt_per_1k_hh"]])
        built.append("supermkt_per_1k_hh")
else:
    print("[NB4] SKIP supermarkets: file missing.")

# Public transport density (NaPTAN)
nt_path = INTERIM / "naptan_stops.parquet"
if nt_path.exists():
    nt = pd.read_parquet(nt_path)
    if "LA_code" in nt.columns:
        stops = nt.groupby("LA_code")[nt.columns[0]].size().rename("stops_count").reset_index()
        print(f"[NB4] OK naptan: used LA_code → {len(stops)} LAs, {int(stops['stops_count'].sum())} stops.")
    else:
        lat_guess, lon_guess = _guess_latlon(nt)
        stops = poi_to_la_count(nt_path, id_col=nt.columns[0], la_code_col=None,
                                lat_col=lat_guess, lon_col=lon_guess, label="naptan")
        stops = stops.rename(columns={"count":"stops_count"})
    if not stops.empty:
        tmp = la_base.merge(stops, on="LA_code", how="left")
        tmp["stops_per_km2"]    = safe_rate(tmp["stops_count"], tmp["area_km2"], 1.0)
        tmp["stops_per_10k_hh"] = safe_rate(tmp["stops_count"], tmp["households_2021"], 10_000.0)
        parts.append(tmp[["LA_code","stops_per_km2","stops_per_10k_hh"]])
        built.extend(["stops_per_km2","stops_per_10k_hh"])
else:
    print("[NB4] SKIP naptan: file missing.")

# IMD (mean score per LA)
imd_path = INTERIM / "imd_lsoa.parquet"
if imd_path.exists():
    imd = pd.read_parquet(imd_path)
    if {"LA_code","imd_score"}.issubset(imd.columns):
        imd_la = (imd.groupby("LA_code")["imd_score"].mean()
                  .rename("imd_score_mean").reset_index())
        parts.append(imd_la)
        built.append("imd_score_mean")
        print(f"[NB4] OK imd: aggregated to {len(imd_la)} LAs.")
    else:
        print("[NB4] SKIP imd: needs LA_code & imd_score.")
else:
    print("[NB4] SKIP imd: file missing.")

# Airbnb per 10k households
air_path = INTERIM / "insideairbnb_listings_city_geo.parquet"
if air_path.exists():
    ab = pd.read_parquet(air_path)
    if "LA_code" in ab.columns:
        ab_la = (ab.groupby("LA_code")[ab.columns[0]].size()
                 .rename("airbnb_count").reset_index())
        print(f"[NB4] OK airbnb: used LA_code → {len(ab_la)} LAs, {int(ab_la['airbnb_count'].sum())} listings.")
    else:
        lat_guess, lon_guess = _guess_latlon(ab)
        ab_la = poi_to_la_count(air_path, id_col=ab.columns[0], la_code_col=None,
                                lat_col=lat_guess, lon_col=lon_guess, label="airbnb")
        ab_la = ab_la.rename(columns={"count":"airbnb_count"})
    if not ab_la.empty:
        tmp = la_base.merge(ab_la, on="LA_code", how="left")
        tmp["airbnb_per_10k_hh"] = safe_rate(tmp["airbnb_count"], tmp["households_2021"], 10_000.0)
        parts.append(tmp[["LA_code","airbnb_per_10k_hh"]])
        built.append("airbnb_per_10k_hh")
else:
    print("[NB4] SKIP airbnb: file missing.")

# Assemble + standardize + summary
if parts:
    static = parts[0].copy()
    for p in parts[1:]:
        static = static.merge(p, on="LA_code", how="outer")

    # Winsorize + z-scores for metric columns
    metric_cols = [c for c in static.columns if c not in {"LA_code"}]
    for c in metric_cols:
        static[c] = winsorize(static[c])
        static[f"{c}_z"] = zscore(static[c])

    # Keep household/area for reference
    static = la_base.merge(static, on="LA_code", how="left")

    static_out = CONTEXT / "context_static.parquet"
    static.to_parquet(static_out, index=False)

    built_sorted = ", ".join(built) if built else "(none)"
    print(f"Saved STATIC context → {static_out} (rows={len(static):,}, cols={len(static.columns)})")
    print(f"[NB4] Built metrics: {built_sorted}")
else:
    print("[NB4] No static context created (no inputs produced counts).")


[NB4] OK fast_food: spatial join via geometry column → 319 LAs, 45654 points.
[NB4] OK supermarkets: spatial join via geometry column → 318 LAs, 10238 points.
[NB4] OK naptan: spatial join via geometry column → 350 LAs, 342793 points.
[NB4] OK imd: aggregated to 317 LAs.
[NB4] OK airbnb: spatial join via geometry column → 44 LAs, 107121 points.
Saved STATIC context → ../data/interim/context/context_static.parquet (rows=318, cols=16)
[NB4] Built metrics: fastfood_per_1k_hh, supermkt_per_1k_hh, stops_per_km2, stops_per_10k_hh, imd_score_mean, airbnb_per_10k_hh


## Save Context Features

In [12]:
time_parts = []

# National private rent index (IPHRP)
iphrp_path = INTERIM / "iphrp_england_monthly.parquet"
if iphrp_path.exists():
    ip = pd.read_parquet(iphrp_path)
    date_col = "month" if "month" in ip.columns else ("date" if "date" in ip.columns else None)
    val_col  = "index_value" if "index_value" in ip.columns else (
        ip.columns.difference([date_col]).tolist()[0] if date_col else None
    )
    if date_col and val_col:
        ip = ip[[date_col, val_col]].rename(columns={date_col:"month", val_col:"iphrp_england"})
        ip["month"] = month_endify(ip["month"])
        ip = ip.sort_values("month").drop_duplicates("month")
        ip["iphrp_yoy"] = ip["iphrp_england"].pct_change(12)

        # Cartesian expand to all LAs
        la_frame = la_base[["LA_code"]].assign(key=1)
        mo_frame = ip.assign(key=1)
        ip_la = la_frame.merge(mo_frame, on="key").drop(columns="key")
        time_parts.append(ip_la[["LA_code","month","iphrp_england","iphrp_yoy"]])
    else:
        print("[NB4][WARN] iphrp_england_monthly schema unexpected — skip.")
else:
    print("[NB4][WARN] iphrp_england_monthly.parquet missing — no time macro.")

# Assemble + standardize
if time_parts:
    time_all = time_parts[0].copy()
    for t in time_parts[1:]:
        time_all = time_all.merge(t, on=["LA_code","month"], how="outer")
    time_all = time_all.sort_values(["LA_code","month"])

    for c in [c for c in time_all.columns if c not in ("LA_code","month")]:
        time_all[c] = winsorize(time_all[c])
        time_all[f"{c}_z"] = zscore(time_all[c])

    time_out = CONTEXT / "context_time.parquet"
    time_all.to_parquet(time_out, index=False)
    print(f"Saved TIME context → {time_out} (rows={len(time_all):,}, cols={len(time_all.columns)})")
else:
    print("[NB4] No time context created.")

Saved TIME context → ../data/interim/context/context_time.parquet (rows=41,022, cols=6)
